# Hierarchical Residual Shape Screen

Stage 3 of the hierarchical modeling track. This notebook tests shrinkage-controlled residual-shape corrections over evaluation progress, boundary distance, and slope-instability coordinates after simple contextual well offsets failed.

## 1. Setup

In [1]:
"Configure imports, paths, and residual-shape priors."

from pathlib import Path
import json
import shutil

import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.linear_model import HuberRegressor, Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures, SplineTransformer, StandardScaler

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "context").exists():
    PROJECT_ROOT = next(parent for parent in PROJECT_ROOT.parents if (parent / "context").exists())

DATE_STAMP = "2026-06-23"
RUN_ID = f"{DATE_STAMP}__hierarchical_residual_shape_screen"

ARTIFACT_TABLES = PROJECT_ROOT / "artifacts" / "tables"
ARTIFACT_REPORTS = PROJECT_ROOT / "artifacts" / "reports"
ARTIFACT_LOGS = PROJECT_ROOT / "artifacts" / "logs" / "runs" / RUN_ID
for path in [ARTIFACT_TABLES, ARTIFACT_REPORTS, ARTIFACT_LOGS]:
    path.mkdir(parents=True, exist_ok=True)

RESIDUAL_PATH = ARTIFACT_TABLES / "model__hsgp_residual_on_gam_huber_residual_sample__2026-06-22.csv"
TARGET_CANDIDATE = "huber_plus_hsgp_dps_m8"
LOCAL_REFERENCE_RMSE = 14.3586
PUBLIC_REFERENCE_RMSE = 14.514
OFFSET_CLIP = 12.0
ROW_FEATURES = ["eval_progress", "distance_log1p", "abs_projected_delta_10"]

print(f"project_root={PROJECT_ROOT}")
print(f"run_id={RUN_ID}")

project_root=C:\Users\sardo\OneDrive\Desktop\Classes\Projects\Wellbore-tvt-Modeling
run_id=2026-06-23__hierarchical_residual_shape_screen


## 2. Load Residual Sample

The residual target is `y_true - y_pred` from the current Huber + additive Hilbert residual model. Every correction is fit on other folds and added to the current prediction.

In [2]:
"Load the promoted candidate residual sample and construct row-shape coordinates."

residuals_all = pd.read_csv(RESIDUAL_PATH)
residuals = residuals_all.loc[residuals_all["candidate"].eq(TARGET_CANDIDATE)].copy()
for col in ["fold", "row_idx", "y_true", "y_pred", "residual", "eval_progress", "distance_from_boundary_md", "abs_projected_delta_10"]:
    residuals[col] = pd.to_numeric(residuals[col], errors="coerce")
residuals["distance_log1p"] = np.log1p(residuals["distance_from_boundary_md"].clip(lower=0))
residuals["baseline_pred"] = residuals["y_pred"]
residuals["baseline_residual"] = residuals["y_true"] - residuals["baseline_pred"]

print(f"rows={len(residuals):,}")
print(f"wells={residuals['wellname'].nunique():,}")
print(residuals.groupby("fold")["wellname"].nunique().rename("wells_by_fold").to_string())

rows=5,400
wells=773
fold
1    258
2    258
3    257


## 3. Candidate Residual-Shape Arms

In [3]:
"Define binned empirical-Bayes and regularized residual-shape candidate arms."

CANDIDATES = [
    {"name": "baseline_no_shape", "kind": "baseline", "shrink": 0.0, "prior_strength": 0.0, "alpha": 0.0, "prior": "zero residual-shape correction"},
    {"name": "progress_bins_p50_s025", "kind": "single_bin", "feature": "eval_progress", "shrink": 0.25, "prior_strength": 50.0, "alpha": 0.0, "prior": "progress-bin means shrunk by 50 pseudo-rows and 75 percent to zero"},
    {"name": "progress_bins_p200_s025", "kind": "single_bin", "feature": "eval_progress", "shrink": 0.25, "prior_strength": 200.0, "alpha": 0.0, "prior": "progress-bin means shrunk by 200 pseudo-rows and 75 percent to zero"},
    {"name": "distance_bins_p100_s025", "kind": "single_bin", "feature": "distance_log1p", "shrink": 0.25, "prior_strength": 100.0, "alpha": 0.0, "prior": "boundary-distance-bin means shrunk toward zero"},
    {"name": "slope_instability_bins_p100_s025", "kind": "single_bin", "feature": "abs_projected_delta_10", "shrink": 0.25, "prior_strength": 100.0, "alpha": 0.0, "prior": "slope-instability-bin means shrunk toward zero"},
    {"name": "additive_bins_p100_s025", "kind": "additive_bins", "shrink": 0.25, "prior_strength": 100.0, "alpha": 0.0, "prior": "additive progress/distance/slope bin deviations with strong shrinkage"},
    {"name": "progress_distance_bins_p100_s025", "kind": "two_dim_bin", "shrink": 0.25, "prior_strength": 100.0, "alpha": 0.0, "prior": "2D progress-distance cell means with strong shrinkage"},
    {"name": "row_linear_ridge_a100_s025", "kind": "ridge", "shrink": 0.25, "prior_strength": 0.0, "alpha": 100.0, "prior": "regularized linear row-shape residual with 75 percent zero shrinkage"},
    {"name": "row_poly2_ridge_a300_s025", "kind": "poly2_ridge", "shrink": 0.25, "prior_strength": 0.0, "alpha": 300.0, "prior": "quadratic row-shape residual with strong coefficient shrinkage"},
    {"name": "progress_spline_ridge_a300_s025", "kind": "progress_spline", "shrink": 0.25, "prior_strength": 0.0, "alpha": 300.0, "prior": "smooth progress residual shape with strong coefficient shrinkage"},
    {"name": "row_huber_s025", "kind": "huber", "shrink": 0.25, "prior_strength": 0.0, "alpha": 0.05, "prior": "robust linear row-shape residual with 75 percent zero shrinkage"},
]

BIN_SPECS = {
    "eval_progress": np.linspace(0.0, 1.000001, 9),
    "distance_log1p": np.linspace(0.0, 10.0, 9),
    "abs_projected_delta_10": np.array([0.0, 2.0, 5.0, 10.0, 20.0, 40.0, 80.0, 160.0, np.inf]),
}

def rmse(y_true, y_pred):
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))

def clipped(values):
    return np.clip(np.asarray(values, dtype=float), -OFFSET_CLIP, OFFSET_CLIP)

print(pd.DataFrame(CANDIDATES)[["name", "kind", "shrink", "prior_strength", "alpha", "prior"]])

                                name             kind  shrink  prior_strength  \
0                  baseline_no_shape         baseline    0.00             0.0   
1             progress_bins_p50_s025       single_bin    0.25            50.0   
2            progress_bins_p200_s025       single_bin    0.25           200.0   
3            distance_bins_p100_s025       single_bin    0.25           100.0   
4   slope_instability_bins_p100_s025       single_bin    0.25           100.0   
5            additive_bins_p100_s025    additive_bins    0.25           100.0   
6   progress_distance_bins_p100_s025      two_dim_bin    0.25           100.0   
7         row_linear_ridge_a100_s025            ridge    0.25             0.0   
8          row_poly2_ridge_a300_s025      poly2_ridge    0.25             0.0   
9    progress_spline_ridge_a300_s025  progress_spline    0.25             0.0   
10                    row_huber_s025            huber    0.25             0.0   

     alpha                 

## 4. Out-Of-Fold Evaluation

In [4]:
"Fit each residual-shape arm on other folds and score the held-out fold."

def assign_bins(values, edges):
    return pd.cut(values, bins=edges, labels=False, include_lowest=True, right=False)

def single_bin_prediction(train_rows, valid_rows, feature, prior_strength):
    train_bins = assign_bins(train_rows[feature], BIN_SPECS[feature])
    valid_bins = assign_bins(valid_rows[feature], BIN_SPECS[feature])
    stats = train_rows.assign(_bin=train_bins).groupby("_bin")["baseline_residual"].agg(["mean", "count"])
    global_mean = float(train_rows["baseline_residual"].mean())
    correction = []
    for bin_id in valid_bins:
        if pd.isna(bin_id) or bin_id not in stats.index:
            correction.append(global_mean * 0.25)
            continue
        count = float(stats.loc[bin_id, "count"])
        mean = float(stats.loc[bin_id, "mean"])
        pooled = (count / (count + prior_strength)) * mean
        correction.append(pooled)
    return np.asarray(correction, dtype=float)

def additive_bin_prediction(train_rows, valid_rows, prior_strength):
    global_mean = float(train_rows["baseline_residual"].mean())
    correction = np.repeat(global_mean, len(valid_rows)).astype(float)
    for feature in ["eval_progress", "distance_log1p", "abs_projected_delta_10"]:
        train_bins = assign_bins(train_rows[feature], BIN_SPECS[feature])
        valid_bins = assign_bins(valid_rows[feature], BIN_SPECS[feature])
        stats = train_rows.assign(_bin=train_bins).groupby("_bin")["baseline_residual"].agg(["mean", "count"])
        deviations = []
        for bin_id in valid_bins:
            if pd.isna(bin_id) or bin_id not in stats.index:
                deviations.append(0.0)
                continue
            count = float(stats.loc[bin_id, "count"])
            mean = float(stats.loc[bin_id, "mean"])
            pooled_deviation = (count / (count + prior_strength)) * (mean - global_mean)
            deviations.append(pooled_deviation)
        correction = correction + np.asarray(deviations, dtype=float)
    return correction

def two_dim_bin_prediction(train_rows, valid_rows, prior_strength):
    p_train = assign_bins(train_rows["eval_progress"], BIN_SPECS["eval_progress"])
    d_train = assign_bins(train_rows["distance_log1p"], BIN_SPECS["distance_log1p"])
    p_valid = assign_bins(valid_rows["eval_progress"], BIN_SPECS["eval_progress"])
    d_valid = assign_bins(valid_rows["distance_log1p"], BIN_SPECS["distance_log1p"])
    global_mean = float(train_rows["baseline_residual"].mean())
    keyed_train = train_rows.assign(_p=p_train, _d=d_train)
    stats = keyed_train.groupby(["_p", "_d"])["baseline_residual"].agg(["mean", "count"])
    correction = []
    for p_bin, d_bin in zip(p_valid, d_valid):
        key = (p_bin, d_bin)
        if pd.isna(p_bin) or pd.isna(d_bin) or key not in stats.index:
            correction.append(global_mean * 0.25)
            continue
        count = float(stats.loc[key, "count"])
        mean = float(stats.loc[key, "mean"])
        pooled = (count / (count + prior_strength)) * mean
        correction.append(pooled)
    return np.asarray(correction, dtype=float)

def model_prediction(train_rows, valid_rows, candidate):
    if candidate["kind"] == "ridge":
        model = Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler()), ("model", Ridge(alpha=candidate["alpha"]))])
        model.fit(train_rows[ROW_FEATURES], train_rows["baseline_residual"])
        return model.predict(valid_rows[ROW_FEATURES])
    if candidate["kind"] == "poly2_ridge":
        model = Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler()), ("poly", PolynomialFeatures(degree=2, include_bias=False)), ("model", Ridge(alpha=candidate["alpha"]))])
        model.fit(train_rows[ROW_FEATURES], train_rows["baseline_residual"])
        return model.predict(valid_rows[ROW_FEATURES])
    if candidate["kind"] == "progress_spline":
        model = Pipeline([("imputer", SimpleImputer(strategy="median")), ("spline", SplineTransformer(n_knots=6, degree=3, include_bias=False)), ("model", Ridge(alpha=candidate["alpha"]))])
        model.fit(train_rows[["eval_progress"]], train_rows["baseline_residual"])
        return model.predict(valid_rows[["eval_progress"]])
    if candidate["kind"] == "huber":
        model = Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler()), ("model", HuberRegressor(alpha=candidate["alpha"], epsilon=1.35, max_iter=500))])
        model.fit(train_rows[ROW_FEATURES], train_rows["baseline_residual"])
        return model.predict(valid_rows[ROW_FEATURES])
    raise ValueError(candidate["kind"])

predicted_rows = []
fold_metric_rows = []
for candidate in CANDIDATES:
    for fold in sorted(residuals["fold"].dropna().unique()):
        train_rows = residuals.loc[~residuals["fold"].eq(fold)].copy()
        valid_rows = residuals.loc[residuals["fold"].eq(fold)].copy()
        raw_correction = np.zeros(len(valid_rows), dtype=float)
        if candidate["kind"] == "single_bin":
            raw_correction = single_bin_prediction(train_rows, valid_rows, candidate["feature"], candidate["prior_strength"])
        elif candidate["kind"] == "additive_bins":
            raw_correction = additive_bin_prediction(train_rows, valid_rows, candidate["prior_strength"])
        elif candidate["kind"] == "two_dim_bin":
            raw_correction = two_dim_bin_prediction(train_rows, valid_rows, candidate["prior_strength"])
        elif candidate["kind"] in ["ridge", "poly2_ridge", "progress_spline", "huber"]:
            raw_correction = model_prediction(train_rows, valid_rows, candidate)
        correction = candidate["shrink"] * clipped(raw_correction)
        y_pred_shape = valid_rows["baseline_pred"].to_numpy(dtype=float) + correction
        eval_frame = valid_rows[["fold", "wellname", "row_idx", "y_true", "baseline_pred", "baseline_residual", "eval_progress", "distance_log1p", "abs_projected_delta_10"]].copy()
        eval_frame["candidate"] = candidate["name"]
        eval_frame["raw_shape_correction"] = raw_correction
        eval_frame["shape_correction"] = correction
        eval_frame["y_pred"] = y_pred_shape
        eval_frame["residual_after_shape"] = eval_frame["y_true"] - eval_frame["y_pred"]
        predicted_rows.append(eval_frame)
        fold_metric_rows.append(
            {
                "candidate": candidate["name"],
                "fold": int(fold),
                "kind": candidate["kind"],
                "shrink": candidate["shrink"],
                "prior_strength": candidate["prior_strength"],
                "alpha": candidate["alpha"],
                "prior": candidate["prior"],
                "n_rows": int(len(valid_rows)),
                "n_wells": int(valid_rows["wellname"].nunique()),
                "rmse": rmse(valid_rows["y_true"], y_pred_shape),
                "mae": float(mean_absolute_error(valid_rows["y_true"], y_pred_shape)),
                "mean_shape_correction": float(np.mean(correction)),
                "shape_correction_std": float(np.std(correction)),
                "mean_residual_after_shape": float(np.mean(eval_frame["residual_after_shape"])),
            }
        )

predictions = pd.concat(predicted_rows, ignore_index=True)
fold_metrics = pd.DataFrame(fold_metric_rows)
summary = (
    fold_metrics.groupby(["candidate", "kind", "shrink", "prior_strength", "alpha", "prior"], as_index=False)
    .agg(
        rmse_mean=("rmse", "mean"),
        rmse_std=("rmse", "std"),
        mae_mean=("mae", "mean"),
        mean_abs_fold_bias=("mean_residual_after_shape", lambda s: float(np.mean(np.abs(s)))),
        mean_shape_correction_abs=("mean_shape_correction", lambda s: float(np.mean(np.abs(s)))),
        shape_correction_std_mean=("shape_correction_std", "mean"),
    )
)
sample_baseline_rmse = float(summary.loc[summary["candidate"].eq("baseline_no_shape"), "rmse_mean"].iloc[0])
summary["rmse_delta_vs_sample_baseline"] = summary["rmse_mean"] - sample_baseline_rmse
summary["rmse_delta_vs_full_reference"] = summary["rmse_mean"] - LOCAL_REFERENCE_RMSE
summary = summary.sort_values("rmse_mean")
summary

,candidate,kind,shrink,prior_strength,alpha,prior,rmse_mean,rmse_std,mae_mean,mean_abs_fold_bias,mean_shape_correction_abs,shape_correction_std_mean,rmse_delta_vs_sample_baseline,rmse_delta_vs_full_reference
8,row_linear_ridge_a100_s025,ridge,0.25,0.0,100.00,regularized linear row-shape residual with 75 ...,14.738241,0.491208,10.413727,0.306233,0.067749,0.150740,-0.000304,0.379641
1,baseline_no_shape,baseline,0.00,0.0,0.00,zero residual-shape correction,14.738545,0.487053,10.413072,0.303161,0.000000,0.000000,0.000000,0.379945
2,distance_bins_p100_s025,single_bin,0.25,100.0,0.00,boundary-distance-bin means shrunk toward zero,14.738555,0.486241,10.415290,0.306369,0.056975,0.042779,0.000010,0.379955
6,progress_spline_ridge_a300_s025,progress_spline,0.25,0.0,300.00,smooth progress residual shape with strong coe...,14.738564,0.486030,10.415863,0.306617,0.068575,0.027035,0.000019,0.379964
7,row_huber_s025,huber,0.25,0.0,0.05,robust linear row-shape residual with 75 perce...,14.739073,0.487650,10.419377,0.366571,0.082484,0.096898,0.000528,0.380473
3,progress_bins_p200_s025,single_bin,0.25,200.0,0.00,progress-bin means shrunk by 200 pseudo-rows a...,14.739807,0.485710,10.416498,0.306299,0.045687,0.065516,0.001262,0.381207
4,progress_bins_p50_s025,single_bin,0.25,50.0,0.00,progress-bin means shrunk by 50 pseudo-rows an...,14.740298,0.485263,10.417707,0.308413,0.058848,0.085121,0.001753,0.381698
5,progress_distance_bins_p100_s025,two_dim_bin,0.25,100.0,0.00,2D progress-distance cell means with strong sh...,14.741339,0.485282,10.417620,0.301947,0.052857,0.087131,0.002794,0.382739
9,row_poly2_ridge_a300_s025,poly2_ridge,0.25,0.0,300.00,quadratic row-shape residual with strong coeff...,14.744284,0.493354,10.419985,0.306428,0.068212,0.173710,0.005739,0.385684
10,slope_instability_bins_p100_s025,single_bin,0.25,100.0,0.00,slope-instability-bin means shrunk toward zero,14.749005,0.494436,10.421099,0.295135,0.063293,0.143464,0.010460,0.390405


## 5. Diagnostics And Decision

In [5]:
"Summarize the best residual-shape arm and well-level residual changes."

best_candidate = summary.iloc[0]["candidate"]
best_row = summary.iloc[0]
baseline_row = summary.loc[summary["candidate"].eq("baseline_no_shape")].iloc[0]
best_predictions = predictions.loc[predictions["candidate"].eq(best_candidate)].copy()
well_after = (
    best_predictions.groupby(["fold", "wellname"], as_index=False)
    .agg(
        n_rows=("residual_after_shape", "size"),
        mean_residual_after_shape=("residual_after_shape", "mean"),
        mae_after_shape=("residual_after_shape", lambda s: float(np.mean(np.abs(s)))),
        rmse_after_shape=("residual_after_shape", lambda s: float(np.sqrt(np.mean(np.asarray(s) ** 2)))),
        mean_shape_correction=("shape_correction", "mean"),
        shape_correction_std=("shape_correction", "std"),
    )
)
well_before = (
    residuals.groupby(["fold", "wellname"], as_index=False)
    .agg(
        mean_residual_before=("baseline_residual", "mean"),
        rmse_before=("baseline_residual", lambda s: float(np.sqrt(np.mean(np.asarray(s) ** 2)))),
    )
)
well_diagnostics = well_after.merge(well_before, on=["fold", "wellname"], how="left")
well_diagnostics["abs_mean_residual_before"] = well_diagnostics["mean_residual_before"].abs()
well_diagnostics["abs_mean_residual_after"] = well_diagnostics["mean_residual_after_shape"].abs()
well_diagnostics["abs_bias_delta"] = well_diagnostics["abs_mean_residual_after"] - well_diagnostics["abs_mean_residual_before"]

findings = pd.DataFrame(
    [
        {
            "finding": "best_candidate",
            "value": f"{best_candidate} RMSE {best_row['rmse_mean']:.4f} vs sample baseline {baseline_row['rmse_mean']:.4f}",
            "modeling_implication": "Promote only if the same-sample gain clears the hierarchy gate.",
        },
        {
            "finding": "sample_baseline_delta",
            "value": f"best candidate delta {best_row['rmse_delta_vs_sample_baseline']:.4f}",
            "modeling_implication": "Negative means row-shape correction improved the same residual frame.",
        },
        {
            "finding": "well_bias_change",
            "value": f"median absolute well-bias delta {well_diagnostics['abs_bias_delta'].median():.4f}",
            "modeling_implication": "Residual-shape correction should not reduce RMSE by creating new well-level bias.",
        },
    ]
)

print(f"best_candidate={best_candidate}")
print(summary.head(12).to_string(index=False))

best_candidate=row_linear_ridge_a100_s025
                       candidate            kind  shrink  prior_strength  alpha                                                                 prior  rmse_mean  rmse_std  mae_mean  mean_abs_fold_bias  mean_shape_correction_abs  shape_correction_std_mean  rmse_delta_vs_sample_baseline  rmse_delta_vs_full_reference
      row_linear_ridge_a100_s025           ridge    0.25             0.0 100.00  regularized linear row-shape residual with 75 percent zero shrinkage  14.738241  0.491208 10.413727            0.306233                   0.067749                   0.150740                      -0.000304                      0.379641
               baseline_no_shape        baseline    0.00             0.0   0.00                                        zero residual-shape correction  14.738545  0.487053 10.413072            0.303161                   0.000000                   0.000000                       0.000000                      0.379945
         d

## 6. Export Artifacts

In [6]:
"Export residual-shape metrics, diagnostics, report, and run bundle."

fold_metrics_path = ARTIFACT_TABLES / f"model__hierarchical_residual_shape_fold_metrics__{DATE_STAMP}.csv"
summary_path = ARTIFACT_TABLES / f"model__hierarchical_residual_shape_summary__{DATE_STAMP}.csv"
residual_sample_path = ARTIFACT_TABLES / f"model__hierarchical_residual_shape_residual_sample__{DATE_STAMP}.csv"
well_diagnostics_path = ARTIFACT_TABLES / f"model__hierarchical_residual_shape_well_diagnostics__{DATE_STAMP}.csv"
findings_path = ARTIFACT_TABLES / f"model__hierarchical_residual_shape_findings__{DATE_STAMP}.csv"
comparison_path = ARTIFACT_TABLES / f"model__hierarchical_residual_shape_comparison__{DATE_STAMP}.csv"
report_path = ARTIFACT_REPORTS / f"model__hierarchical_residual_shape_screen__{DATE_STAMP}.md"

fold_metrics.to_csv(fold_metrics_path, index=False)
summary.to_csv(summary_path, index=False)
best_predictions.to_csv(residual_sample_path, index=False)
well_diagnostics.to_csv(well_diagnostics_path, index=False)
findings.to_csv(findings_path, index=False)
comparison = pd.DataFrame(
    [
        {
            "date": DATE_STAMP,
            "model": best_candidate,
            "local_rmse": float(best_row["rmse_mean"]),
            "local_rmse_std": float(best_row["rmse_std"]),
            "delta_vs_sample_baseline": float(best_row["rmse_delta_vs_sample_baseline"]),
            "delta_vs_full_huber_hilbert_reference": float(best_row["rmse_delta_vs_full_reference"]),
            "public_lb_reference": PUBLIC_REFERENCE_RMSE,
            "decision": "promote_to_next_hierarchy" if best_row["rmse_delta_vs_sample_baseline"] <= -0.05 else "do_not_promote_yet",
            "notes": "Residual-shape correction on sampled out-of-fold residuals; no Kaggle submission.",
        }
    ]
)
comparison.to_csv(comparison_path, index=False)

def markdown_table(frame):
    display_frame = frame.copy()
    for col in display_frame.select_dtypes(include=["float"]).columns:
        display_frame[col] = display_frame[col].map(lambda value: "" if pd.isna(value) else f"{value:.4f}")
    columns = list(display_frame.columns)
    header = "| " + " | ".join(columns) + " |"
    divider = "| " + " | ".join(["---"] * len(columns)) + " |"
    rows = []
    for _, row in display_frame.iterrows():
        rows.append("| " + " | ".join(str(row[col]) for col in columns) + " |")
    return "\n".join([header, divider] + rows)

report_lines = [
    "# Hierarchical Residual Shape Screen",
    "",
    f"Date: {DATE_STAMP}",
    "",
    "## Purpose",
    "",
    "Test shrinkage-controlled residual-shape corrections over evaluation progress, boundary distance, and slope-instability coordinates.",
    "",
    "## Priors And Assumptions",
    "",
    f"- Corrections are residual-shape offsets centered on zero and clipped to +/- {OFFSET_CLIP:.1f} TVT units before shrinkage.",
    "- Binned arms use empirical-Bayes count shrinkage toward zero or global residual shape.",
    "- Regularized arms use Ridge/Huber penalties and fixed shrinkage before scoring.",
    "- Every arm trains on other folds only and scores the held-out fold.",
    "",
    "## Results",
    "",
    markdown_table(summary.head(12)),
    "",
    "## Findings",
    "",
    markdown_table(findings),
    "",
    "## Decision",
    "",
    f"Best candidate `{best_candidate}` scored **{best_row['rmse_mean']:.4f}** RMSE, delta **{best_row['rmse_delta_vs_sample_baseline']:.4f}** vs the sample no-shape baseline and **{best_row['rmse_delta_vs_full_reference']:.4f}** vs the stored full Huber + Hilbert reference.",
    "",
    "## Artifacts",
    "",
    f"- Fold metrics: `{fold_metrics_path}`",
    f"- Summary: `{summary_path}`",
    f"- Residual sample: `{residual_sample_path}`",
    f"- Well diagnostics: `{well_diagnostics_path}`",
    f"- Findings: `{findings_path}`",
    f"- Comparison row: `{comparison_path}`",
]
report_path.write_text("\n".join(report_lines) + "\n", encoding="utf-8")

for path in [fold_metrics_path, summary_path, residual_sample_path, well_diagnostics_path, findings_path, comparison_path, report_path]:
    shutil.copy2(path, ARTIFACT_LOGS / path.name)
metadata = {
    "run_id": RUN_ID,
    "best_candidate": str(best_candidate),
    "best_rmse": float(best_row["rmse_mean"]),
    "best_delta_vs_sample_baseline": float(best_row["rmse_delta_vs_sample_baseline"]),
    "best_delta_vs_full_reference": float(best_row["rmse_delta_vs_full_reference"]),
    "reference_rmse": LOCAL_REFERENCE_RMSE,
    "public_reference_rmse": PUBLIC_REFERENCE_RMSE,
    "decision": str(comparison.iloc[0]["decision"]),
}
(ARTIFACT_LOGS / "metadata.json").write_text(json.dumps(metadata, indent=2), encoding="utf-8")

print(report_path)
print(summary_path)
print(fold_metrics_path)

C:\Users\sardo\OneDrive\Desktop\Classes\Projects\Wellbore-tvt-Modeling\artifacts\reports\model__hierarchical_residual_shape_screen__2026-06-23.md
C:\Users\sardo\OneDrive\Desktop\Classes\Projects\Wellbore-tvt-Modeling\artifacts\tables\model__hierarchical_residual_shape_summary__2026-06-23.csv
C:\Users\sardo\OneDrive\Desktop\Classes\Projects\Wellbore-tvt-Modeling\artifacts\tables\model__hierarchical_residual_shape_fold_metrics__2026-06-23.csv


## Results

This notebook is a residual-shape hierarchy screen. It determines whether row-level shape corrections should be promoted before any full hierarchical HSGP work.